## 0. Project About

- Dataset: CheXpert Plus.
- Variabel Masukan: Citra Chest X-Ray.
- Variabel Target: `Findings` dan `Impression`.
- Paradigma Pembelajaran: *Supervised Learning*.
- Tugas Spesifik Pembelajaran Mesin: VLM $-$ *Medical Report Generation*.
* Permasalahan utama VLM medis saat ini:
  * kebutuhan komputasi tinggi.
  * risiko inkonsistensi klinis.
  * error propagation pada pendekatan single-sequence autoregressive.
* Penelitian mengusulkan arsitektur Decoupled Dual-Head Decoder berbasis Small Language Model (SLM) dengan optimasi QLoRA 4-bit.
* Mekanisme dual-head:
  * Head Findings menghasilkan deskripsi detail berdasarkan fitur visual citra.
  * Head Impression menghasilkan kesimpulan diagnosis berdasarkan fitur visual dan findings.
* Dataset menggunakan CheXpert Plus subset data.
* Komponen model:
  * Vision Encoder: BiomedCLIP (frozen weights).
  * Language Decoder: Qwen2.5-7B-Instruct dengan dual-LoRA adapters.

## 1. Import Library

In [1]:
import numpy as np
import pandas as pd
from skmultilearn.model_selection import IterativeStratification
from IPython.display import display
import json
import os
from pathlib import Path
import re
import random
import torch

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

## 2. Load Dataset

In [2]:
# define dataset path
BASE_DIR = Path("D:/VLM-Research_Task-C/data/ChexPert/vlm_fah/dataset/chexpertplus")
CSV_PATH = Path("D:/VLM-Research_Task-C/data/ChexPert/df_chexpert_plus_240401.csv")

In [3]:
# load metadata into dataframe
df = pd.read_csv(CSV_PATH)

In [4]:
# preview metadata
df.head()

,path_to_image,path_to_dcm,frontal_lateral,ap_pa,deid_patient_id,patient_report_date_order,report,section_narrative,section_clinical_history,section_history,...,section_accession_number,age,sex,race,ethnicity,interpreter_needed,insurance_type,recent_bmi,deceased,split
0,train/patient00003/study1/view1_frontal.jpg,train/patient00003/study1/view1_frontal.dcm,Frontal,AP,patient00003,1,"NARRATIVE:\nCHEST, ONE VIEW: 2-10-2001\nFINDIN...","\nCHEST, ONE VIEW: 2-10-2001\n",NaN,NaN,...,\nLFEWOZWVDRX\nThis report has been anonymized...,41.0,Male,White,Non-Hispanic/Non-Latino,Unknown,Private Insurance,NaN,No,train
1,train/patient00007/study2/view1_frontal.jpg,train/patient00007/study2/view1_frontal.dcm,Frontal,AP,patient00007,2,NARRATIVE:\nChest 1 View: July 20\n \nHISTORY:...,\nChest 1 View: July 20\n \n,NaN,"Male, 69 years old, intubated.\n \n",...,\n485K2I4588\nThis report has been anonymized....,69.0,Male,Other,Hispanic/Latino,No,Private Insurance,NaN,No,train
2,train/patient00007/study1/view1_frontal.jpg,train/patient00007/study1/view1_frontal.dcm,Frontal,AP,patient00007,1,NARRATIVE:\nChest 1 View: 12-28-2000\n \nHISTO...,\nChest 1 View: 12-28-2000\n \n,NaN,"Male, 69 years old, Check tube placement.\n \n",...,\nRN\nThis report has been anonymized. All dat...,69.0,Male,Other,Hispanic/Latino,No,Private Insurance,NaN,No,train
3,train/patient00009/study1/view2_lateral.jpg,train/patient00009/study1/view2_lateral.dcm,Lateral,NaN,patient00009,1,NARRATIVE:\nCHEST X-RAY: 2/21/2006\nCOMPARISON...,\nCHEST X-RAY: 2/21/2006\n,Dyspnea. Multiple myeloma.\n,NaN,...,\n6340977630\nThis report has been anonymized....,76.0,Male,Asian,Non-Hispanic/Non-Latino,Unknown,Medicare,NaN,No,train
4,train/patient00009/study1/view1_frontal.jpg,train/patient00009/study1/view1_frontal.dcm,Frontal,PA,patient00009,2,NARRATIVE:\nCHEST X-RAY: 4/27/07\nCOMPARISON: ...,\nCHEST X-RAY: 4/27/07\n,Dyspnea. Multiple myeloma.\n,NaN,...,\n#624515709\nThis report has been anonymized....,76.0,Male,Asian,Non-Hispanic/Non-Latino,Unknown,Medicare,NaN,No,train


In [5]:
# list of columns of the metadata
df.columns

Index(['path_to_image', 'path_to_dcm', 'frontal_lateral', 'ap_pa',
       'deid_patient_id', 'patient_report_date_order', 'report',
       'section_narrative', 'section_clinical_history', 'section_history',
       'section_comparison', 'section_technique', 'section_procedure_comments',
       'section_findings', 'section_impression', 'section_end_of_impression',
       'section_summary', 'section_accession_number', 'age', 'sex', 'race',
       'ethnicity', 'interpreter_needed', 'insurance_type', 'recent_bmi',
       'deceased', 'split'],
      dtype='object')

In [6]:
# filter columns needed
columns_to_keep = [
    'path_to_image', 
    'split', 
    'section_findings', 
    'section_impression', 
]
df = df[columns_to_keep].copy()

In [7]:
# load clinical label from JSON
df_labels = pd.read_json(BASE_DIR / "label/impression_fixed.json", lines=True)

# merge dataset: text + pathological label
df_merged = pd.merge(df, df_labels, on='path_to_image', how='inner')

In [8]:
# synchronize image path
def fix_image_path(row):
    # .jpg to .png
    pure_path = Path(row['path_to_image']).with_suffix('.png')
    # adjust path
    full_path = BASE_DIR / "preprocessed" / "PNG" / pure_path
    return str(full_path)
df_merged['actual_image_path'] = df_merged.apply(fix_image_path, axis=1)

In [9]:
# dataset size based on categorical column on 'split'
print(df_merged['split'].value_counts())

split
train    223228
valid       234
Name: count, dtype: int64


In [10]:
# preview main structure of dataset
df_merged[['actual_image_path', 'section_findings', 'section_impression']].head()

,actual_image_path,section_findings,section_impression
0,D:\VLM-Research_Task-C\data\ChexPert\vlm_fah\d...,"Costophrenic angles sharp, without evidence o...",\n1. NO EVIDENCE OF PNEUMOTHORAX.\n2. MILD INT...
1,D:\VLM-Research_Task-C\data\ChexPert\vlm_fah\d...,NaN,\n \nLow lung volumes. Stable moderate enlar...
2,D:\VLM-Research_Task-C\data\ChexPert\vlm_fah\d...,NaN,"\n \nLow lung volumes, and overlying trauma b..."
3,D:\VLM-Research_Task-C\data\ChexPert\vlm_fah\d...,NaN,\n1. PA AND LATERAL CHEST RADIOGRAPH. THE HEAR...
4,D:\VLM-Research_Task-C\data\ChexPert\vlm_fah\d...,NaN,\n1. PA AND LATERAL CHEST RADIOGRAPH. THE HEAR...


In [11]:
# preview full text of findings and impression neatly

# show full text without truncation
pd.set_option('display.max_colwidth', None)

# create preview dataframe
preview_df = df_merged[['section_findings', 'section_impression']].head(15)

# reset index for cleaner display
preview_df = preview_df.reset_index(drop=True)

# display neatly in notebook
display(preview_df)

,section_findings,section_impression
0,"Costophrenic angles sharp, without evidence of effusion.\nThe cardiomediastinal silhouette is normal. Vessels mildly\nindistinct with prominence of interstitial structures, suggesting\nmild, pulmonary edema. Left subclavian central venous catheter is\nseen, tip in mid SVC. No pneumothorax.\n",\n1. NO EVIDENCE OF PNEUMOTHORAX.\n2. MILD INTERSTITIAL PULMONARY EDEMA.\n
1,NaN,"\n \nLow lung volumes. Stable moderate enlargement of the \ncardiomediastinal silhouette. Normal pulmonary vascularity. Patchy \nopacity at the left lung base, likely atelectasis. No pleural \neffusion or pneumothorax. No fractures identified.\n \nEndotracheal tube tip is in the mid trachea, overlying the T4 \nvertebral body.\n \n"
2,NaN,"\n \nLow lung volumes, and overlying trauma board limit evaluation. There \nis prominence of the cardiac shadow, likely reflecting cardiomegaly, \nand retrocardiac opacity, likely representing atelectasis. No \ndefinite pleural fluid. The left costophrenic sulcus is not \nincluded, limiting evaluation for pneumothorax.\n \nEndotracheal tube tip is in the mid trachea, overlying the T4 \nvertebral body.\n \nNo fracture identified.\n \n"
3,NaN,"\n1. PA AND LATERAL CHEST RADIOGRAPH. THE HEART IS ENLARGED, WITH\nA CTR OF 16/29. NO CEPHALIZATION OR OVERT PULMONARY EDEMA. LINEAR\nSHADOWING IS SEEN AT BOTH LUNG BASES, SUGGESTING ATELECTASIS, MOST\nMARKED ON THE RIGHT. ON THE LATERAL VIEW, THIS APPEARS TO LIE\nWITHIN THE RIGHT MIDDLE LOBE. THE LUNG APICES APPEAR CLEAR.\n2. DISC DEGENERATION IS SEEN THROUGHOUT THE THORACIC SPINE,\nWITHOUT SIGNIFICANT VERTEBRAL BODY COLLAPSE.\n"
4,NaN,"\n1. PA AND LATERAL CHEST RADIOGRAPH. THE HEART IS ENLARGED, WITH\nA CTR OF 16/29. NO CEPHALIZATION OR OVERT PULMONARY EDEMA. LINEAR\nSHADOWING IS SEEN AT BOTH LUNG BASES, SUGGESTING ATELECTASIS, MOST\nMARKED ON THE RIGHT. ON THE LATERAL VIEW, THIS APPEARS TO LIE\nWITHIN THE RIGHT MIDDLE LOBE. THE LUNG APICES APPEAR CLEAR.\n2. DISC DEGENERATION IS SEEN THROUGHOUT THE THORACIC SPINE,\nWITHOUT SIGNIFICANT VERTEBRAL BODY COLLAPSE.\n"
5,NaN,\n \n1. ACUTE TO SUBACUTE LEFT POSTERIOR SIXTH RIB FRACTURE.\n \n2.NO EVIDENCE OF PNEUMOTHORAX. MINIMAL LEFT PLEURAL THICKENING MAY \nREPRESENT SMALL AMOUNT OF BLOOD IN THE PLEURAL SPACE.\n \n3.SUBTLE OPACITY JUST LATERAL TO THE CARDIAC APEX IN THE LEFT BASE \nMAY REPRESENT A SMALL AREA OF PULMONARY CONTUSION OR OTHER CAUSE FOR \nSMALL FOCUS OF CONSOLIDATION.\n \n4.RIGHT LUNG IS CLEAR.\n \n5.LOW LUNG VOLUMES.\n \n
6,NaN,\n \n1. ACUTE TO SUBACUTE LEFT POSTERIOR SIXTH RIB FRACTURE.\n \n2.NO EVIDENCE OF PNEUMOTHORAX. MINIMAL LEFT PLEURAL THICKENING MAY \nREPRESENT SMALL AMOUNT OF BLOOD IN THE PLEURAL SPACE.\n \n3.SUBTLE OPACITY JUST LATERAL TO THE CARDIAC APEX IN THE LEFT BASE \nMAY REPRESENT A SMALL AREA OF PULMONARY CONTUSION OR OTHER CAUSE FOR \nSMALL FOCUS OF CONSOLIDATION.\n \n4.RIGHT LUNG IS CLEAR.\n \n5.LOW LUNG VOLUMES.\n \n
7,NaN,\n1. NO FOCAL CONSOLIDATION OR EVIDENCE OF LYMPHADENOPATHY. NORMAL\nCHEST.\n
8,\nLow lung volumes. Discoid atelectasis and consolidation seen in\nthe left lower lobe with an elevated left hemidiaphragm. This is\nunchanged from the previous chest x-ray.\n,\nDISCOID CONSOLIDATION AND ATELECTASIS OF THE LEFT LOWER LOBE.\nUNCHANGED FROM THE PREVIOUS CHEST X-RAY.\n
9,\nLow lung volumes. Discoid atelectasis and consolidation seen in\nthe left lower lobe with an elevated left hemidiaphragm. This is\nunchanged from the previous chest x-ray.\n,\nDISCOID CONSOLIDATION AND ATELECTASIS OF THE LEFT LOWER LOBE.\nUNCHANGED FROM THE PREVIOUS CHEST X-RAY.\n


## 3. EDA and Initial Cleaning

### 3.1 Identify and Handle Missing Values

In [12]:
print("Missing Findings:", df_merged['section_findings'].isna().sum())
print("Missing Impression:", df_merged['section_impression'].isna().sum())

Missing Findings: 163993
Missing Impression: 145


In [13]:
# handle missing value
df_cleaned = df_merged.copy()
df_cleaned['section_findings'] = df_merged['section_findings'].fillna("")
df_cleaned['section_impression'] = df_merged['section_impression'].fillna("")

# filter
df_cleaned = df_cleaned[(df_cleaned['section_findings'].str.strip().str.len() > 0) | (df_cleaned['section_impression'].str.strip().str.len() > 0)].copy() 

### 3.2 Check for Duplicate

In [14]:
dup_count = df_cleaned.duplicated(subset=['actual_image_path', 'section_findings', 'section_impression']).sum()
print("jumlah baris duplikat:", dup_count)

jumlah baris duplikat: 0


### 3.3 Token Length Analysis

In [15]:
df_cleaned['findings_words'] = df_cleaned['section_findings'].apply(lambda x: len(str(x).split()))
df_cleaned['impression_words'] = df_cleaned['section_impression'].apply(lambda x: len(str(x).split()))

print("\nDeskripsi Panjang Kata Findings:")
print(df_cleaned['findings_words'].describe())
print("\nDeskripsi Panjang Kata Impression:")
print(df_cleaned['impression_words'].describe())


Deskripsi Panjang Kata Findings:
count    223335.000000
mean         16.191448
std          32.646268
min           0.000000
25%           0.000000
50%           0.000000
75%          21.000000
max         509.000000
Name: findings_words, dtype: float64

Deskripsi Panjang Kata Impression:
count    223335.000000
mean         41.799364
std          21.939881
min           0.000000
25%          27.000000
50%          38.000000
75%          53.000000
max         425.000000
Name: impression_words, dtype: float64


*Inference*: Mayoritas teks (75%) panjanganya masih di bawah 128.

### 3.4 Text Structure Identification and Cleaning

In [16]:
# preview full text of findings and impression neatly

# show full text without truncation
pd.set_option('display.max_colwidth', None)

# create preview dataframe
preview_df = df_cleaned[['section_findings', 'section_impression']].head(15)

# reset index for cleaner display
preview_df = preview_df.reset_index(drop=True)

# display neatly in notebook
display(preview_df)

,section_findings,section_impression
0,"Costophrenic angles sharp, without evidence of effusion.\nThe cardiomediastinal silhouette is normal. Vessels mildly\nindistinct with prominence of interstitial structures, suggesting\nmild, pulmonary edema. Left subclavian central venous catheter is\nseen, tip in mid SVC. No pneumothorax.\n",\n1. NO EVIDENCE OF PNEUMOTHORAX.\n2. MILD INTERSTITIAL PULMONARY EDEMA.\n
1,,"\n \nLow lung volumes. Stable moderate enlargement of the \ncardiomediastinal silhouette. Normal pulmonary vascularity. Patchy \nopacity at the left lung base, likely atelectasis. No pleural \neffusion or pneumothorax. No fractures identified.\n \nEndotracheal tube tip is in the mid trachea, overlying the T4 \nvertebral body.\n \n"
2,,"\n \nLow lung volumes, and overlying trauma board limit evaluation. There \nis prominence of the cardiac shadow, likely reflecting cardiomegaly, \nand retrocardiac opacity, likely representing atelectasis. No \ndefinite pleural fluid. The left costophrenic sulcus is not \nincluded, limiting evaluation for pneumothorax.\n \nEndotracheal tube tip is in the mid trachea, overlying the T4 \nvertebral body.\n \nNo fracture identified.\n \n"
3,,"\n1. PA AND LATERAL CHEST RADIOGRAPH. THE HEART IS ENLARGED, WITH\nA CTR OF 16/29. NO CEPHALIZATION OR OVERT PULMONARY EDEMA. LINEAR\nSHADOWING IS SEEN AT BOTH LUNG BASES, SUGGESTING ATELECTASIS, MOST\nMARKED ON THE RIGHT. ON THE LATERAL VIEW, THIS APPEARS TO LIE\nWITHIN THE RIGHT MIDDLE LOBE. THE LUNG APICES APPEAR CLEAR.\n2. DISC DEGENERATION IS SEEN THROUGHOUT THE THORACIC SPINE,\nWITHOUT SIGNIFICANT VERTEBRAL BODY COLLAPSE.\n"
4,,"\n1. PA AND LATERAL CHEST RADIOGRAPH. THE HEART IS ENLARGED, WITH\nA CTR OF 16/29. NO CEPHALIZATION OR OVERT PULMONARY EDEMA. LINEAR\nSHADOWING IS SEEN AT BOTH LUNG BASES, SUGGESTING ATELECTASIS, MOST\nMARKED ON THE RIGHT. ON THE LATERAL VIEW, THIS APPEARS TO LIE\nWITHIN THE RIGHT MIDDLE LOBE. THE LUNG APICES APPEAR CLEAR.\n2. DISC DEGENERATION IS SEEN THROUGHOUT THE THORACIC SPINE,\nWITHOUT SIGNIFICANT VERTEBRAL BODY COLLAPSE.\n"
5,,\n \n1. ACUTE TO SUBACUTE LEFT POSTERIOR SIXTH RIB FRACTURE.\n \n2.NO EVIDENCE OF PNEUMOTHORAX. MINIMAL LEFT PLEURAL THICKENING MAY \nREPRESENT SMALL AMOUNT OF BLOOD IN THE PLEURAL SPACE.\n \n3.SUBTLE OPACITY JUST LATERAL TO THE CARDIAC APEX IN THE LEFT BASE \nMAY REPRESENT A SMALL AREA OF PULMONARY CONTUSION OR OTHER CAUSE FOR \nSMALL FOCUS OF CONSOLIDATION.\n \n4.RIGHT LUNG IS CLEAR.\n \n5.LOW LUNG VOLUMES.\n \n
6,,\n \n1. ACUTE TO SUBACUTE LEFT POSTERIOR SIXTH RIB FRACTURE.\n \n2.NO EVIDENCE OF PNEUMOTHORAX. MINIMAL LEFT PLEURAL THICKENING MAY \nREPRESENT SMALL AMOUNT OF BLOOD IN THE PLEURAL SPACE.\n \n3.SUBTLE OPACITY JUST LATERAL TO THE CARDIAC APEX IN THE LEFT BASE \nMAY REPRESENT A SMALL AREA OF PULMONARY CONTUSION OR OTHER CAUSE FOR \nSMALL FOCUS OF CONSOLIDATION.\n \n4.RIGHT LUNG IS CLEAR.\n \n5.LOW LUNG VOLUMES.\n \n
7,,\n1. NO FOCAL CONSOLIDATION OR EVIDENCE OF LYMPHADENOPATHY. NORMAL\nCHEST.\n
8,\nLow lung volumes. Discoid atelectasis and consolidation seen in\nthe left lower lobe with an elevated left hemidiaphragm. This is\nunchanged from the previous chest x-ray.\n,\nDISCOID CONSOLIDATION AND ATELECTASIS OF THE LEFT LOWER LOBE.\nUNCHANGED FROM THE PREVIOUS CHEST X-RAY.\n
9,\nLow lung volumes. Discoid atelectasis and consolidation seen in\nthe left lower lobe with an elevated left hemidiaphragm. This is\nunchanged from the previous chest x-ray.\n,\nDISCOID CONSOLIDATION AND ATELECTASIS OF THE LEFT LOWER LOBE.\nUNCHANGED FROM THE PREVIOUS CHEST X-RAY.\n


In [17]:
PATTERNS = {
    # Non-clinical communication noise
    'review_noise': re.compile(
        r'(?:i have personally reviewed the images|agreed\s*with the report transcribed above)', 
        re.IGNORECASE
    ),
    'call_logs': re.compile(
        r'results called to\s+[\w\s,]+(?:\s+at\s+\d+)?(?:\s*hours)?', 
        re.IGNORECASE
    ),
    
    # Section enumeration/bullet points
    'numbering': re.compile(r'(?:\\n|\n|^)\d+\.\s*'),
    
    # Corrupted inline newlines (e.g., "is\nseen" or "right\nncostophrenic")
    'broken_newlines': re.compile(r'(?:\\n|\n)[nN]?(?=[a-zA-Z])'),
    'standalone_newlines': re.compile(r'\\n|\n'),
    
    # Administrative metadata and section labels
    'meta_id_date': re.compile(r'#\d+:\s*\d+[-/]\d+[-/]\d+'),
    'hours_suffix': re.compile(r'\b\d{4}\s*hours\b', re.IGNORECASE),
    'headers': re.compile(r'\b(?:findings|impression|section|examination):\s*', re.IGNORECASE),
    
    # Whitespace normalization
    'spaces': re.compile(r'\s+')
}

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # Apply regex transformations
    text = PATTERNS['review_noise'].sub('', text)
    text = PATTERNS['call_logs'].sub('', text)
    text = PATTERNS['numbering'].sub(' ', text)
    text = PATTERNS['broken_newlines'].sub('', text) 
    text = PATTERNS['standalone_newlines'].sub(' ', text)
    text = PATTERNS['meta_id_date'].sub('', text)
    text = PATTERNS['hours_suffix'].sub('', text)
    text = PATTERNS['headers'].sub('', text)
    # Final whitespace trimming
    text = PATTERNS['spaces'].sub(' ', text)
    return text.strip()

In [18]:
# clean section_findings
df_cleaned['cleaned_findings'] = df_cleaned['section_findings'].fillna("").astype(str).apply(clean_text)

# clean section_impression
df_cleaned['cleaned_impression'] = df_cleaned['section_impression'].fillna("").astype(str).apply(clean_text)

# sanity check
print("=== SAMPLE HASIL CLEANING ===")
sample_row = df_cleaned.dropna(subset=['section_findings', 'section_impression']).iloc[3]

print("\n[ORIGINAL FINDINGS]:\n", sample_row['section_findings'])
print("-> [CLEANED FINDINGS]:\n", sample_row['cleaned_findings'])

print("\n[ORIGINAL IMPRESSION]:\n", sample_row['section_impression'])
print("-> [CLEANED IMPRESSION]:\n", sample_row['cleaned_impression'])

=== SAMPLE HASIL CLEANING ===

[ORIGINAL FINDINGS]:
 
-> [CLEANED FINDINGS]:
 

[ORIGINAL IMPRESSION]:
 
1. PA AND LATERAL CHEST RADIOGRAPH. THE HEART IS ENLARGED, WITH
A CTR OF 16/29. NO CEPHALIZATION OR OVERT PULMONARY EDEMA. LINEAR
SHADOWING IS SEEN AT BOTH LUNG BASES, SUGGESTING ATELECTASIS, MOST
MARKED ON THE RIGHT. ON THE LATERAL VIEW, THIS APPEARS TO LIE
WITHIN THE RIGHT MIDDLE LOBE. THE LUNG APICES APPEAR CLEAR.
2. DISC DEGENERATION IS SEEN THROUGHOUT THE THORACIC SPINE,
WITHOUT SIGNIFICANT VERTEBRAL BODY COLLAPSE.

-> [CLEANED IMPRESSION]:
 PA AND LATERAL CHEST RADIOGRAPH. THE HEART IS ENLARGED, WITHA CTR OF 16/29. NO CEPHALIZATION OR OVERT PULMONARY EDEMA. LINEARSHADOWING IS SEEN AT BOTH LUNG BASES, SUGGESTING ATELECTASIS, MOSTMARKED ON THE RIGHT. ON THE LATERAL VIEW, THIS APPEARS TO LIEWITHIN THE RIGHT MIDDLE LOBE. THE LUNG APICES APPEAR CLEAR. DISC DEGENERATION IS SEEN THROUGHOUT THE THORACIC SPINE,WITHOUT SIGNIFICANT VERTEBRAL BODY COLLAPSE.


In [19]:
def clean_garbage_text(text):
    if pd.isna(text):
        return ""
    
    text_str = str(text).strip()
    
    if re.match(r'^[^a-zA-Z0-9]*$', text_str):
        return ""

    garbage_words = {
        # Konjungsi & Preposisi
        "and", "or", "with", "for", "to", "the", "an", "a", "of", "at", "by", "from", "in", "on",
        # Kata ganti / Penunjuk sisa
        "this", "that", "these", "those", "it", "is", "was", "were",
        # Artefak teks medis / Simbol tekstual tunggal
        "null", "none", "unspecified", "history", "examination", "finding", "findings", "impression"
    }
    
    # Jika teks hanya terdiri dari satu kata dan ada di dalam kamus sampah
    if text_str.lower() in garbage_words:
        return ""
    
    return text_str

In [20]:
# bersihkan kata sampah menjadi string kosong
df_cleaned['cleaned_findings'] = df_cleaned['cleaned_findings'].apply(clean_garbage_text)
df_cleaned['cleaned_impression'] = df_cleaned['cleaned_impression'].apply(clean_garbage_text)

# drop baris jika KEDUA kolom kosong
df_cleaned = df_cleaned[~((df_cleaned['cleaned_findings'] == "") & (df_cleaned['cleaned_impression'] == ""))].copy()

# isi sisa string kosong dengan teks default klinis
df_cleaned['cleaned_findings'] = df_cleaned['cleaned_findings'].replace("", "No significant visual findings.")
df_cleaned['cleaned_impression'] = df_cleaned['cleaned_impression'].replace("", "No definitive clinical impression.")

## 4. Data Splitting

In [21]:
print(df_cleaned['split'].value_counts())

split
train    223092
valid       234
Name: count, dtype: int64


In [22]:
print("=== STARTING FULL-SCALE MULTI-LABEL STRATIFICATION PIPELINE ===")

# 14 standard clinical labels from dataset CheXpert Plus
diseases = [
    "Enlarged Cardiomediastinum", 
    "Cardiomegaly", 
    "Lung Opacity", 
    "Lung Lesion", 
    "Edema", 
    "Consolidation", 
    "Pneumonia", 
    "Atelectasis", 
    "Pneumothorax", 
    "Pleural Effusion", 
    "Pleural Other", 
    "Fracture", 
    "Support Devices", 
    "No Finding"
]

# =====================================================================
#         FINAL DATA PREPARATION & UNCERTAINTY HANDLING
# =====================================================================
# Standarisasi nilai label (Handle NaN -> 0.0, Handle Uncertain -1.0 -> 0.0 / U-Zero Approach)
df_cleaned[diseases] = df_cleaned[diseases].fillna(0.0).replace(-1.0, 0.0).astype(int)

# Urutkan index secara eksplisit agar susunan baris selalu konstan di setiap session
df_cleaned = df_cleaned.sort_index()

X = df_cleaned.index.values 
y = df_cleaned[diseases].values

# =========================================================================
#  STAGE 1 SPLITTING: Ambil 85% untuk Train Set, Sisa 15% untuk Remainder
# =========================================================================
print("\nExecuting Stage 1 Splitting (Train vs Remainder)... Please wait...")
stratifier_1 = IterativeStratification(
    n_splits=2, 
    order=1, 
    sample_distribution_per_fold=[0.15, 0.85]  # Fold 0: 15% (Rest), Fold 1: 85% (Train)
)

# .split() langsung mengembalikan tuple array berisi indeks baris
train_indices_actual, rest_indices_actual = next(stratifier_1.split(X, y))

# Memastikan array yang lebih besar (85%) masuk ke variabel train
if len(train_indices_actual) < len(rest_indices_actual):
    train_indices_actual, rest_indices_actual = rest_indices_actual, train_indices_actual

df_train = df_cleaned.iloc[train_indices_actual].copy()
df_rest = df_cleaned.iloc[rest_indices_actual].copy()

# =========================================================================
#  STAGE 2 SPLITTING: Bagi Sisa 15% Menjadi 50% Valid Set dan 50% Test Set
# =========================================================================
print("Executing Stage 2 Splitting (Validation vs Test)...")
X_rest = df_rest.index.values
y_rest = df_rest[diseases].values

stratifier_2 = IterativeStratification(
    n_splits=2, 
    order=1, 
    sample_distribution_per_fold=[0.5, 0.5]  # Dibagi adil rata tengah (50:50)
)

# Langsung tangkap array indeks 50:50
valid_indices_actual, test_indices_actual = next(stratifier_2.split(X_rest, y_rest))
df_valid = df_rest.iloc[valid_indices_actual].copy()
df_test = df_rest.iloc[test_indices_actual].copy()

# =====================================================================
#         LOG CONFIGURATION & DISTRIBUTION VALIDATION
# =====================================================================
print("\n==================================================")
print("     MULTI-LABEL STRATIFICATION SUCCESS REPORT    ")
print("==================================================")
print(f" Total Size of Dataset  : {len(df_cleaned)} samples")
print(f" Data Train Final (85%)    : {len(df_train)} samples")
print(f" Data Validation Final (7.5%): {len(df_valid)} samples")
print(f" Data Test Final (7.5%)    : {len(df_test)} samples")
print("==================================================")

# Hitung distribusi untuk memastikan keseimbangan 14 patologi
dist_check = pd.DataFrame({
    'Train (%)': df_train[diseases].mean() * 100,
    'Valid (%)': df_valid[diseases].mean() * 100,
    'Test (%)': df_test[diseases].mean() * 100
})
print("\n=== DISTRIBUSI KESEIMBANGAN KELAS PATOLOGI (%) ===")
print(dist_check.round(2))

=== STARTING FULL-SCALE MULTI-LABEL STRATIFICATION PIPELINE ===

Executing Stage 1 Splitting (Train vs Remainder)... Please wait...
Executing Stage 2 Splitting (Validation vs Test)...

     MULTI-LABEL STRATIFICATION SUCCESS REPORT    
 Total Size of Dataset  : 223326 samples
 Data Train Final (85%)    : 189827 samples
 Data Validation Final (7.5%): 16750 samples
 Data Test Final (7.5%)    : 16749 samples

=== DISTRIBUSI KESEIMBANGAN KELAS PATOLOGI (%) ===
                            Train (%)  Valid (%)  Test (%)
Enlarged Cardiomediastinum       3.38       3.39      3.39
Cardiomegaly                    13.68      13.68     13.68
Lung Opacity                    46.10      46.10     46.10
Lung Lesion                      4.19       4.19      4.20
Edema                           23.74      23.74     23.74
Consolidation                    6.14       6.13      6.14
Pneumonia                        2.17       2.17      2.17
Atelectasis                     15.16      15.16     15.16
Pneumoth

In [23]:
# ==========================================================================
#    EXPORTING DATA TO TARGET  DIRECTORY (D:\VLM-Research_Task-C\data)
# ==========================================================================
print("\nExporting split datasets to secure drive D... Please wait...")
target_dir = r"D:\VLM-Research_Task-C\data"
if not os.path.exists(target_dir):
    os.makedirs(target_dir)
    print(f"-> Created new directory path at: {target_dir}")

train_path = os.path.join(target_dir, "chexpert_train_split.csv")
valid_path = os.path.join(target_dir, "chexpert_valid_split.csv")
test_path  = os.path.join(target_dir, "chexpert_test_split.csv")

df_train.to_csv(train_path, index=False)
df_valid.to_csv(valid_path, index=False)
df_test.to_csv(test_path, index=False)

print("\n==================================================")
print("  ASET DATA RISET MURNI BERHASIL DIKUNCI PERMANEN! ")
print("==================================================")
print(f" Saved Train File Path      : {train_path}")
print(f" Saved Validation File Path : {valid_path}")
print(f" Saved Test File Path       : {test_path}")
print("==================================================")


Exporting split datasets to secure drive D... Please wait...

  ASET DATA RISET MURNI BERHASIL DIKUNCI PERMANEN! 
 Saved Train File Path      : D:\VLM-Research_Task-C\data\chexpert_train_split.csv
 Saved Validation File Path : D:\VLM-Research_Task-C\data\chexpert_valid_split.csv
 Saved Test File Path       : D:\VLM-Research_Task-C\data\chexpert_test_split.csv
